# LAB- P2: El Modelo de Overshooting de Dornbusch en Tiempo Discreto (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P2-v1.0
*   **Capítulo de Referencia**: Capítulo 3, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar el comportamiento dinámico de una pequeña economía abierta con movilidad perfecta de capitales. Estudiar cómo responde el tipo de cambio nominal a shocks monetarios y por qué experimenta una sobrerreacción o *overshooting* inicial debido a la rigidez temporal de los precios nacionales.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo de transmisión de la política monetaria en una economía abierta y cómo interactúan el mercado de bienes y el mercado de dinero.
2.  **Visualizar** el fenómeno de *overshooting* del tipo de cambio y su posterior convergencia dinámica.
3.  **Identificar** y analizar sistemas dinámicos con estabilidad de **punto de silla** (saddle point) en tiempo discreto.
4.  **Representar** e interpretar la transición de una economía en un **Diagrama de Fases** bidimensional que contenga curvas de demarcación, el camino de silla estable y el campo de vectores.


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

> *   **📋 Antes de empezar**, consulta ' (en esta misma carpeta): objetivos, tiempo estimado y conocimientos previos de esta práctica.
> 
> ⏱️ **~90-120 minutos**
> 
> 📋 **Prerrequisitos**: **Matemáticas**: sistemas de ecuaciones en diferencias, autovalores y punto de silla, diagramas de fases en tiempo discreto. | **Economía**: paridad descubierta de intereses (UIP), modelo IS-LM estático, neutralidad monetaria a largo plazo, relaci...
> 
### 🕹️ GUÍA RÁPIDA DE INICIO - Overshooting de Dornbusch
*   **¿Qué estamos haciendo aquí?** Analizamos cómo responde el tipo de cambio (el valor de la moneda) ante un shock.
*   **La gran idea (Sobrerreacción):** Como los precios de los supermercados tardan mucho en cambiar (son rígidos), el mercado financiero (tipo de cambio) reacciona con un "salto" exagerado al principio para compensar.
*   **¡Prueba esto!** Cambia la oferta monetaria `m0` en la simulación y observa la línea del tipo de cambio dar un salto enorme antes de estabilizarse lentamente en su nuevo valor.


In [ ]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


## Extensiones para ABP (Aprendizaje Basado en Proyectos)

1. **Shock fiscal**: simular un aumento del gasto público ($\beta_0$) y analizar si produce overshooting o undershooting, y por qué.
2. **Ajuste gradual de expectativas**: modificar la UIP para que las expectativas cambiarias sean adaptativas en vez de racionales (ej. $\Delta s^e = \lambda(s^* - s)$) y comparar la dinámica.
3. **Calibración con datos reales**: calibrar $\mu$ y $\nu$ con datos trimestrales de inflación y brecha de producción de una economía real (ej. España 1999-2019) y simular un shock de política monetaria.

In [ ]:
# "using X" trae a este cuaderno todo el código público del paquete X, para
# no tener que reescribirlo. Pkg.activate("../..") usa el entorno del repo
# (Project.toml/Manifest.toml de la raíz). Pkg.instantiate() instala lo que
# falte de ese entorno (la primera vez puede tardar; las siguientes es
# instantáneo).
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

# La lógica del modelo Dornbusch vive en src/models/Dornbusch.jl dentro del
# paquete MacroAIComp. El notebook solo llama funciones ya probadas, no
# reimplementa fórmulas (ver Sección 6 del notebook Python).
using MacroAIComp
using Plots
# "import Plots: mm" trae solo el nombre "mm" (unidad de margen)
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using Interact                 # widgets interactivos (sliders) para Jupyter
using BenchmarkTools           # medición de rendimiento (Fase III)


## 1. El Marco Teórico: Ecuaciones y Estabilidad de Punto de Silla

El modelo de Dornbusch describe una economía pequeña y abierta con perfecta movilidad de capitales bajo las siguientes ecuaciones:

### 1.1 Ecuaciones Estructurales
1.  **Mercado Monetario (Curva LM):**
    $$m_t - p_t = \psi y^n_t - \theta i_t$$
    Donde $m$ es el logaritmo de la oferta monetaria, $p$ el log de precios, $y^n$ el log de producción potencial (asumimos $y_t = y^n_t$ para simplificar) e $i$ el tipo de interés nominal. Despejando el tipo de interés nominal:
    $$i_t = \frac{p_t - m_t + \psi y^n_t}{\theta}$$

2.  **Demanda Agregada en Economía Abierta (Curva IS):**
    $$y^d_t = \beta_0 + \beta_1(s_t - p_t + p^*_t) - \beta_2 i_t$$
    Donde $s$ es el log del tipo de cambio nominal (moneda nacional por moneda extranjera), $p^*$ los precios extranjeros e $y^d$ la demanda agregada. La demanda depende positivamente del tipo de cambio real (competitividad exterior) y negativamente del tipo de interés nominal.

3.  **Ajuste de Precios (Curva de Phillips en diferencias):**
    $$\Delta p_t = p_{t+1} - p_t = \mu(y^d_t - y^n_t)$$
    Donde $\mu$ representa la velocidad de ajuste del mercado de bienes.

4.  **Paridad No Cubierta de Intereses (UIP - Expectativas Racionales):**
    $$\Delta s_t = s_{t+1} - s_t = i_t - i^*_t$$
    Donde $i^*$ es el tipo de interés extranjero. Bajo previsión perfecta, la depreciación esperada del tipo de cambio coincide con la depreciación efectiva $\Delta s_t$.

---

### 1.2 Reducción a un Sistema Dinámico Lineal en Diferencias
Sustituyendo $i_t$ e $y^d_t$ en las ecuaciones de transición de $p_t$ y $s_t$, el sistema dinámico bidimensional se escribe en forma matricial como:
$$\begin{bmatrix} \Delta p_t \\ \Delta s_t \end{bmatrix} = \mathbf{A} \begin{bmatrix} p_t \\ s_t \end{bmatrix} + \mathbf{B} \mathbf{z}_t$$
Donde el vector de variables exógenas es $\mathbf{z}_t = [\beta_0, m_t, y^n_t, i^*_t, p^*_t]^T$ y las matrices son:
$$\mathbf{A} = \begin{bmatrix} -\mu\left( \beta_1 + \frac{\beta_2}{\theta} \right) & \mu\beta_1 \\ \frac{1}{\theta} & 0 \end{bmatrix}$$
$$\mathbf{B} = \begin{bmatrix} \mu & \frac{\mu\beta_{2}}{\theta} & - \mu\left(\frac{\psi\beta_{2}}{\theta} + 1\right) & 0 & \mu\beta_{1} \\ 0 & - \frac{1}{\theta} & \frac{\psi}{\theta} & - 1 & 0 \end{bmatrix}$$

Este sistema dinámico tiene una estructura de **punto de silla** (saddle point): posee un autovalor estable (cuyo módulo de $1+\lambda$ es menor a la unidad) y otro inestable. La variable de precios ($p$) es lenta y rígida a corto plazo, mientras que el tipo de cambio ($s$) es una variable forward-looking flexible que "salta" instantáneamente ante shocks para situar a la economía en la trayectoria de convergencia estable.


In [ ]:
# Esta celda solo FIJA NÚMEROS (Capítulo 3 del libro): todavía no calcula
# nada. default_calibration(DornbuschParams) devuelve un DornbuschParams, un
# struct (definido en src/models/Dornbusch.jl): una "ficha" con 10 campos
# (psi, theta, beta1, beta2, mi, beta0, m0, ypot0, pstar0, istar0). Usar
# default_calibration() evita errores de tecleo: los valores están
# centralizados en el código fuente y testeados. Al ejecutar veremos los 10
# valores impresos con su descripción económica como comprobación visual.
params_sim = default_calibration(DornbuschParams)

# Glosario didáctico: descripción económica y símbolo de cada parámetro técnico
descriptions = Dict(
    "psi" => "Sensibilidad de la demanda de dinero respecto al PIB [ψ]",
    "theta" => "Sensibilidad de la demanda de dinero respecto al interés nominal [θ]",
    "beta1" => "Sensibilidad de la demanda agregada respecto al tipo de cambio real [β1]",
    "beta2" => "Sensibilidad de la demanda agregada respecto al interés nominal [β2]",
    "mi" => "Velocidad de ajuste de precios ante excesos de demanda (Phillips) [μ]",
    "beta0" => "Demanda agregada autónoma base [β0]",
    "m0" => "Oferta monetaria nominal (logaritmo) [M0]",
    "ypot0" => "Producción potencial (pleno empleo) [ypot]",
    "pstar0" => "Logaritmo del nivel de precios extranjero [pstar]",
    "istar0" => "Tipo de interés nominal extranjero (porcentaje) [istar]",
)

println("CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):")
println("="^78)
println(rpad("Variable", 12), " | ", rpad("Valor", 6), " | ", rpad("Descripción Económica", 50))
println("-"^78)
for field in fieldnames(typeof(params_sim))
    name = string(field)
    value = getfield(params_sim, field)
    desc = get(descriptions, name, "Parámetro del modelo")
    println("  ", rpad(name, 10), " | ", rpad(value, 6), " | ", rpad(desc, 50))
end
println("="^78)


## 2. Equilibrio de Largo Plazo (Estado Estacionario)

En el largo plazo, las variables se estabilizan, por lo que las variaciones temporales son nulas ($\Delta p_t = 0$ y $\Delta s_t = 0$). Resolviendo analíticamente:
1.  **De la ecuación de precios:** $\Delta p_t = 0 \Rightarrow y^d_t = y^n_t = \bar{Y}$.
2.  **De la ecuación del tipo de cambio:** $\Delta s_t = 0 \Rightarrow i_t = i^*_t$.
3.  **Del mercado de dinero:** Sustituyendo el interés en la demanda de saldos reales:
    $$p^* = m_t - \psi y^n_t + \theta i^*_t$$
4.  **Del mercado de bienes:** Sustituyendo las condiciones anteriores en la demanda agregada y despejando el tipo de cambio nominal de largo plazo:
    $$s^* = m_t - \frac{\beta_0}{\beta_1} + \left( \frac{1 - \psi\beta_1}{\beta_1} \right) y^n_t + \left( \frac{\theta\beta_1 + \beta_2}{\beta_1} \right) i^*_t - p^*_t$$

*Nota sobre fe errata del libro:* En el texto del Capítulo 3, por un error de imprenta, el denominador de la última fracción de la ecuación del tipo de cambio estacionario se imprime como $\beta_2$ en lugar de $\beta_1$. El código de la biblioteca `macroaicomp` implementa la fórmula matemáticamente correcta con el denominador $\beta_1$, lo que reproduce el valor numérico exacto de equilibrio del libro ($s^* = 76.52$).


In [ ]:
# steady_state() es una FUNCIÓN: le pasamos los parámetros y nos devuelve un
# diccionario con los valores de equilibrio de largo plazo (p, s, i, yd, dp,
# ds). Por dentro resuelve Delta_p=0, Delta_s=0 usando las fórmulas
# analíticas: p* = m - psi*ypot + theta*istar, s* = m - beta0/beta1 +
# ((1-psi*beta1)/beta1)*ypot + ((theta*beta1+beta2)/beta1)*istar - pstar.
# println() imprime texto en la consola (como print() en Python). Al ejecutar
# veremos: p*=1.5, s*=76.515, i*=3.0%.
ss_init = steady_state(params_sim)

println("VALORES DE EQUILIBRIO DE LARGO PLAZO:")
println("  Precios nacionales (p*)    : ", ss_init["p"])
println("  Tipo de cambio nominal (s*): ", ss_init["s"])
println("  Tipo de interés (i*)       : ", ss_init["i"], "%")


## 3. Detrás de la Escena: El Salto del Tipo de Cambio al Camino de Silla Estable

En un sistema dinámico con un punto de silla, si la economía sufre un shock imprevisto, la única forma de evitar que las variables exploten y diverjan hacia el infinito a largo plazo es que la variable flexible (el tipo de cambio, $s$) **salte instantáneamente en el periodo del shock** hacia la trayectoria estable (el *camino de silla* o *stable path*).

### La Ecuación del Salto de Expectativas
La trayectoria de convergencia estable del tipo de cambio nominal está gobernada por el autovalor estable $\lambda_1$ (el que cumple $|1+\lambda_1| < 1$):
$$\Delta s_t = \lambda_1(s_t - \bar{s}_t)$$
Igualando esta condición de convergencia con la ecuación de Paridad No Cubierta de Intereses (UIP) de la economía en el momento del shock ($t=1$), podemos despejar el valor exacto al que debe saltar el tipo de cambio:
$$\lambda_1(s_1 - \bar{s}_1) = -\frac{1}{\theta}(m_1 - p_1 - \psi y^n_1) - i^*_1$$
$$s_1 = \frac{-(m_1 - p_1 - \psi y^n_1)}{\theta \lambda_1} - \frac{i^*_1}{\lambda_1} + \bar{s}_1$$

Dado que los precios son rígidos en el primer período, $p_1$ se mantiene en su nivel antiguo pre-shock. Sin embargo, dado que $\lambda_1$ es negativo (aproximadamente $-0.74$), el tipo de cambio nominal experimenta una **sobrerreacción inicial** ($s_1$ sube de golpe por encima de su nivel de equilibrio final $\bar{s}_1$).


In [ ]:
# Esta celda DEFINE (no ejecuta) simulate_dornbusch_manual(), una versión
# didáctica de simulate_shock() que desglosa paso a paso la recursión en
# diferencias del modelo de Dornbusch. El objetivo es que veas CÓMO funciona
# el salto del tipo de cambio (overshooting): en el periodo del shock, s NO
# sigue la ecuación estándar, sino que salta directamente al "camino de
# silla estable" usando el autovalor estable lambda1. Los precios p son
# rígidos: no se mueven en el periodo del shock. Esta es la esencia del
# modelo: una variable "de salto" (s, forward-looking) y otra
# "predeterminada" (p, lenta). "function nombre(...) end" define una FUNCIÓN
# en Julia, igual que "def" en Python pero terminada con "end".
# = [beta0, m, ypot, istar, pstar] (vector de variables exógenas)
function simulate_dornbusch_manual(params, z_init, z_final, periods=30, shock_period=1)
    # 1. Obtener autovalores de la matriz del sistema para identificar el autovalor estable
    lambdas_m = eigenvalues(params)
    stable_idx_m = argmin(abs.(lambdas_m .+ 1.0))
    stable_lambda_m = lambdas_m[stable_idx_m]  # lambda1 (aprox -0.74)

    # 2. Inicializar arrays de trayectorias
    p = zeros(periods)
    s = zeros(periods)

    # Calcular estados estacionarios
    p_ss_init = z_init[2] - params.psi * z_init[3] + params.theta * z_init[4]
    s_ss_final = (
        z_final[2]
        - z_final[1] / params.beta1
        + ((1.0 - params.psi * params.beta1) / params.beta1) * z_final[3]
        + ((params.theta * params.beta1 + params.beta2) / params.beta1) * z_final[4]
        - z_final[5]
    )

    # Periodo inicial: Estado estacionario inicial
    p[1] = p_ss_init
    s[1] = z_init[2] - z_init[1]/params.beta1 + ((1.0 - params.psi*params.beta1)/params.beta1)*z_init[3] + ((params.theta*params.beta1 + params.beta2)/params.beta1)*z_init[4] - z_init[5]

    # Evolución temporal paso a paso
    for t in 2:periods
        if t == shock_period + 1
            # Precios rígidos a corto plazo
            p[t] = p[t - 1]
            # Salto del tipo de cambio al stable path (Overshooting)
            m_1 = z_final[2]
            ypot_1 = z_final[3]
            istar_1 = z_final[4]
            s[t] = (
                -(m_1 - p[t] - params.psi * ypot_1) / (params.theta * stable_lambda_m)
                - istar_1 / stable_lambda_m
                + s_ss_final
            )
        else
            # Propagación estándar para periodos posteriores usando las ecuaciones del sistema
            z_curr = t >= shock_period + 1 ? z_final : z_init
            i_prev = -(z_curr[2] - p[t-1] - params.psi * z_curr[3]) / params.theta
            yd_prev = z_curr[1] + params.beta1 * (s[t-1] - p[t-1] + z_curr[5]) - params.beta2 * i_prev
            dp_prev = params.mi * (yd_prev - z_curr[3])
            ds_prev = i_prev - z_curr[4]

            p[t] = p[t-1] + dp_prev
            s[t] = s[t-1] + ds_prev
        end
    end

    return p, s
end

# Verificación: comparamos la simulación manual contra el resolvedor de la
# biblioteca para el mismo shock monetario que usa el diagrama de fases.
p_manual, s_manual = simulate_dornbusch_manual(params_sim, [500.0, 100.0, 2000.0, 3.0, 0.0], [500.0, 101.0, 2000.0, 3.0, 0.0], 30, 1)
res_lib = simulate_shock(params_sim, [500.0, 100.0, 2000.0, 3.0, 0.0], [500.0, 101.0, 2000.0, 3.0, 0.0], 30, 1)
max_err_p = maximum(abs.(p_manual .- res_lib["p"]))
max_err_s = maximum(abs.(s_manual .- res_lib["s"]))
println("Función de simulación manual registrada con éxito.")
println("Diferencia máxima vs simulate_shock(): p -> ", max_err_p, ", s -> ", max_err_s)
@assert max_err_p < 1e-8 && max_err_s < 1e-8


In [ ]:
# eigenvalues() devuelve los autovalores de la matriz A del sistema. En el
# modelo de Dornbusch, la matriz A tiene un autovalor estable (lambda1 aprox
# -0.74, cuyo |1+lambda| < 1) y otro inestable (lambda2 aprox 0.54). Esta
# estructura de punto de silla es la que permite el overshooting: el tipo de
# cambio "salta" para colocarse sobre la senda estable. argmin() con el
# broadcasting ".+ 1.0" encuentra el índice del autovalor estable.
lambdas = eigenvalues(params_sim)
println("Autovalores: ", lambdas)
stable_idx = argmin(abs.(lambdas .+ 1.0))
stable_lambda = lambdas[stable_idx]
println("Autovalor estable: ", stable_lambda)


## 4. Transmisión Económica e Interactividad (Diagrama de Fases)

### 4.1 El Mecanismo de Overshooting
Cuando la oferta monetaria se incrementa ($M_0 \uparrow$):
1.  **A corto plazo:** Los precios nacionales son rígidos ($P_1 = P_0$). La liquidez agregada reduce el interés nacional ($i_1 \downarrow$). Como el tipo extranjero $i^*$ es constante, la UIP exige una expectativa de apreciación cambiaria futura. Para que el tipo de cambio pueda apreciarse en el futuro y a la vez converger a un nivel de largo plazo que es más depreciado, el tipo de cambio nominal **debe experimentar un salto vertical inicial sobredimensionado** ($s_1 \uparrow \uparrow$).
2.  **Durante la transición:** El tipo de cambio real depreciado y los tipos bajos expanden la demanda agregada ($Y^d_1 > Y^n$). Esto genera inflación gradual ($\Delta p_t > 0$). Conforme $P$ sube, la liquidez se contrae, $i$ sube de vuelta a $i^*$ y el tipo de cambio nominal se aprecia progresivamente deslizándose por el camino de silla estable.

### 4.2 Locus y Camino de Silla en el Gráfico de Fases
*   **Locus $\Delta s_t = 0$ (Equilibrio de Dinero):** Línea vertical en el nivel de precios estacionario $p_t = p^*$.
*   **Locus $\Delta p_t = 0$ (Equilibrio de Bienes):** Línea con pendiente positiva ($1.01$ en calibración base) dada por:
    $$s_t = \frac{\beta_0 + p^*_t \beta_1 + \frac{\beta_2}{\theta}(m_t - \psi y^n_t) - y^n_t \left(1 + \frac{\psi \beta_2}{\theta}\right)}{\beta_1} + \left(1 + \frac{\beta_2}{\theta \beta_1}\right) p_t$$
*   **Saddle Path (Camino de Silla Estable):** Línea con pendiente negativa ($k \approx -2.70$):
    $$s_t - \bar{s} = k(p_t - \bar{p})$$


In [ ]:
# @manipulate es el equivalente en Julia de interact() de Python: crea
# sliders y redibuja automáticamente cada vez que los mueves. El código
# dentro del bloque simula el modelo de Dornbusch y dibuja 3 paneles:
# (1) trayectorias temporales de s y p, (2) tipo de interés y demanda
# agregada, (3) diagrama de fases en el plano (p, s) con locus, saddle path
# y campo vectorial. Al mover los sliders verás en vivo cómo cambia el
# overshooting del tipo de cambio.
@manipulate for m0_val in slider(98.0:0.5:104.0; value=101.0, label="Dinero (M)"), beta0_val in slider(450.0:10.0:550.0; value=500.0, label="Gasto (B0)")
    
    z_initial = [500.0, 100.0, 2000.0, 3.0, 0.0]
    z_final = [beta0_val, m0_val, 2000.0, 3.0, 0.0]
    periods = 30
    
    # Calcular nuevos parámetros con z_final
    params_final = DornbuschParams(params_sim.theta, params_sim.psi, params_sim.beta1, 
                                 params_sim.beta2, params_sim.mi, 
                                 beta0_val, m0_val, 2000.0, 3.0, 0.0)
    ss_final = steady_state(params_final)
    
    res = simulate_shock(params_sim, z_initial, z_final, periods, 1)
    
    # Panel 1: Dinámica Temporal p y s
    t_axis = 0:(periods-1)
    p1 = plot(t_axis, res["s"], color="#7A3E9F", lw=2.5, label="Tipo cambio (s)")
    plot!(t_axis, res["p"], color="#8EAD3A", lw=2.5, label="Precios (p)")
    hline!([ss_init["s"]], color="#7A3E9F", ls=:dot, label="s Inicial")
    hline!([ss_final["s"]], color="#7A3E9F", ls=:dash, label="s Final")
    hline!([ss_init["p"]], color="#8EAD3A", ls=:dot, label="p Inicial")
    hline!([ss_final["p"]], color="#8EAD3A", ls=:dash, label="p Final")
    vline!([1], color=:red, ls=:dash, label="Shock (t=1)")
    title!("Trayectorias Temporales (s y p)")
    xlabel!("Tiempo (t)")
    ylabel!("Escala Logarítmica")
    
    # Panel 2: Interés y Demanda Agregada
    p2 = plot(t_axis, res["i"], color="#004C97", lw=2.0, label="Interés (i)")
    plot!(t_axis, res["yd"], color="#D95319", lw=2.0, label="Demanda (yd)")
    hline!([z_final[4]], color="#004C97", ls=:dash, label="i*")
    hline!([z_final[3]], color="#D95319", ls=:dash, label="ypot")
    vline!([1], color=:red, ls=:dash, label="")
    title!("Tipos de Interés y Demanda Agregada")
    xlabel!("Tiempo (t)")
    
    # Panel 3: Diagrama de Fases
    p3 = plot(res["p"], res["s"], color="#7A3E9F", lw=3, label="Trayectoria dinámica")
    vline!([ss_final["p"]], color="#004C97", ls=:dash, lw=2, label="ds = 0 (Final)")
    vline!([ss_init["p"]], color="#004C97", ls=:dot, label="ds = 0 (Inicial)")
    
    p_vals = range(minimum(res["p"]) - 0.5, maximum(res["p"]) + 0.5, length=100)
    slope_dp = 1.0 + params_sim.beta2 / (params_sim.theta * params_sim.beta1)
    c_locus_init = ss_init["s"] - slope_dp * ss_init["p"]
    c_locus_final = ss_final["s"] - slope_dp * ss_final["p"]
    
    s_locus_init = c_locus_init .+ slope_dp .* p_vals
    s_locus_final = c_locus_final .+ slope_dp .* p_vals
    
    plot!(p_vals, s_locus_init, color="#8EAD3A", ls=:dot, label="dp = 0 (Inicial)")
    plot!(p_vals, s_locus_final, color="#8EAD3A", ls=:dash, lw=2, label="dp = 0 (Final)")
    
    a_mat, _ = coefficient_matrices(params_sim)
    k_slope = (stable_lambda - a_mat[1,1]) / a_mat[1,2]
    saddle_final = ss_final["s"] .+ k_slope .* (p_vals .- ss_final["p"])
    plot!(p_vals, saddle_final, color=:black, ls=:dashdot, label="Saddle Path")
    
    p_grid = range(minimum(res["p"]) - 0.3, maximum(res["p"]) + 0.3, length=10)
    s_grid = range(minimum(res["s"]) - 0.5, maximum(res["s"]) + 0.5, length=10)
    
    # Normalización uniforme: la longitud de cada flecha es la misma fracción
    # (5%) del rango de su propio eje, evitando una escala arbitraria distinta
    # entre p y s.
    arrow_frac = 0.05
    p_range = maximum(p_grid) - minimum(p_grid)
    s_range = maximum(s_grid) - minimum(s_grid)

    p_pts, s_pts, dp_pts, ds_pts = Float64[], Float64[], Float64[], Float64[]
    for pp in p_grid, ss in s_grid
        i_pt = -(z_final[2] - pp - params_sim.psi * z_final[3]) / params_sim.theta
        yd_pt = z_final[1] + params_sim.beta1 * (ss - pp + z_final[5]) - params_sim.beta2 * i_pt
        dp = params_sim.mi * (yd_pt - z_final[3])
        ds = i_pt - z_final[4]

        norm = sqrt(dp^2 + ds^2)
        if norm > 0
            push!(p_pts, pp); push!(s_pts, ss)
            push!(dp_pts, (dp/norm)*arrow_frac*p_range); push!(ds_pts, (ds/norm)*arrow_frac*s_range)
        end
    end
    quiver!(p_pts, s_pts, quiver=(dp_pts, ds_pts), color=:gray, alpha=0.5)
    
    scatter!([ss_init["p"]], [ss_init["s"]], color=:gray, markersize=6, label="EE Inic")
    scatter!([ss_final["p"]], [ss_final["s"]], color=:black, marker=:star, markersize=8, label="EE Fin")

    # Salto instantáneo del tipo de cambio en el periodo del shock (t=0 -> t=1)
    plot!([res["p"][1], res["p"][2]], [res["s"][1], res["s"][2]], color="#7A3E9F", lw=2, arrow=:closed, label="")
    annotate!(res["p"][1] - 0.15, (res["s"][1] + res["s"][2]) / 2, text("Jump", "#7A3E9F", :bold, 9))

    title!("Plano de Fases (p, s)")
    xlabel!("Precios (p)")
    ylabel!("Tipo de cambio (s)")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Respuesta del modelo Dornbusch", top_margin=10mm)
end


## 5. Verificación frente al oráculo

Comparamos contra los valores reportados en el libro y reproducidos por el
código DYNARE del Apéndice F (`referencia/icm3.mod`), recogidos en
`oraculo.md`:

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Estado estacionario inicial — $p$ | 1.5 |
| Estado estacionario inicial — $s$ | 76.515 |
| Estado estacionario inicial — $i$ | 3.0 |
| Estado estacionario inicial — $y^d$ | 2000.0 |
| Estado estacionario inicial — $\Delta p$ | 0.0 |
| Estado estacionario inicial — $\Delta s$ | 0.0 |
| Autovalor estable $\lambda_1$ | −0.7415 |
| Autovalor inestable $\lambda_2$ | 0.5395 |
| $p$ en $t=1$ (shock $M_0: 100 \to 101$) | 1.5 (sticky) |
| $s$ en $t=1$ (overshooting) | 80.215 |
| $i$ en $t=1$ (cae por shock expansivo) | 1.0 |
| $p$ en largo plazo ($t=29$, nuevo SS) | 2.5 |
| $s$ en largo plazo ($t=29$, nuevo SS) | 77.515 |
| $i$ en largo plazo (vuelve a $i^*$) | 3.0 |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.

In [ ]:
# Verificamos que el estado estacionario, los autovalores y la simulación del
# shock monetario coinciden con el oráculo del Apéndice F (DYNARE) recogido en
# oraculo.md. @assert isapprox compara dos valores y SOLO lanza un error si la
# diferencia supera la tolerancia atol. No usamos "==" porque la aritmética
# con decimales casi nunca da resultados exactamente iguales (errores de
# redondeo internos). Si el port a Julia tuviera un error, esta celda lanzaría
# AssertionError y detendría la ejecución antes de seguir construyendo
# gráficos sobre un resultado incorrecto.

# Verificar estado estacionario
@assert isapprox(ss_init["p"], 1.5; atol=1e-4)
@assert isapprox(ss_init["s"], 76.515; atol=1e-4)
@assert isapprox(ss_init["i"], 3.0; atol=1e-4)
@assert isapprox(ss_init["yd"], 2000.0; atol=1e-4)
@assert isapprox(ss_init["dp"], 0.0; atol=1e-4)
@assert isapprox(ss_init["ds"], 0.0; atol=1e-4)

# Verificar autovalores (punto de silla: uno estable, otro inestable)
lambdas_sorted = sort(lambdas)
@assert isapprox(lambdas_sorted[1], -0.7415; atol=1e-4)
@assert isapprox(lambdas_sorted[2], 0.5395; atol=1e-4)

# Verificar shock monetario (M0: 100 -> 101) en t=1 (índice 2 en Julia)
@assert isapprox(res_lib["p"][2], 1.5; atol=1e-4)    # precios rígidos
@assert isapprox(res_lib["s"][2], 80.215; atol=5e-3)  # overshooting
@assert isapprox(res_lib["i"][2], 1.0; atol=1e-3)     # cae por shock expansivo

# Verificar valores en largo plazo (t=29, índice 30 en Julia)
@assert isapprox(res_lib["p"][30], 2.5; atol=1e-2)
@assert isapprox(res_lib["s"][30], 77.515; atol=1e-2)
@assert isapprox(res_lib["i"][30], 3.0; atol=1e-2)    # vuelve a i*
println("OK: coincide con el oráculo DYNARE (Apéndice F).")


## 6. Buenas Prácticas Aplicadas en este Laboratorio

Fíjate en las siguientes decisiones de diseño técnico que hacen de este código un estándar ejemplar:
1.  **Higiene de Datos de Entrada**: Se aíslan los parámetros estructurales en un dataclass `DornbuschParams` para evitar valores ocultos dentro de las ecuaciones.
2.  **Lógica Externa Reutilizable**: Las rutinas matriciales complejas residen en `src/macroaicomp/models/dornbusch.jl` facilitando el testeo y mantenimiento.
3.  **Higiene de Control de Versiones**: Las celdas de este cuaderno se han limpiado de outputs volátiles mediante `nbstripout` antes de confirmar la versión en el repositorio de código.


## Extensiones para ABP (Aprendizaje Basado en Proyectos)

1. **Shock fiscal**: simular un aumento del gasto público ($\beta_0$) y analizar si produce overshooting o undershooting, y por qué.
2. **Ajuste gradual de expectativas**: modificar la UIP para que las expectativas cambiarias sean adaptativas en vez de racionales (ej. $\Delta s^e = \lambda(s^* - s)$) y comparar la dinámica.
3. **Calibración con datos reales**: calibrar $\mu$ y $\nu$ con datos trimestrales de inflación y brecha de producción de una economía real (ej. España 1999-2019) y simular un shock de política monetaria.

## 8. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# @btime (BenchmarkTools.jl) ejecuta la función muchas veces y muestra el
# tiempo mínimo/medio de ejecución y la memoria asignada. El $ delante de
# las variables evita que BenchmarkTools las trate como globales, lo que
# falsearía la medición (Fase III del proyecto). Al ejecutar veremos cuánto
# tarda Julia en simular 30 periodos del modelo de Dornbusch con salto.
@btime simulate_shock($params_sim, [500.0, 100.0, 2000.0, 3.0, 0.0], [500.0, 101.0, 2000.0, 3.0, 0.0], 30, 1)
